Tomas Stankevičius, 2425049, Ver. 1

# 01 - Data Pipeline: Lithuanian Cultural VLM Dataset

**Goal:** Build an original ~250-image / ~1000-QA-pair Lithuanian cultural dataset for fine-tuning Qwen2-VL-2B.

### Why this approach?
Fine-tuning a Vision-Language Model (VLM) for a specific culture (Lithuanian) requires high-quality, diverse, and accurately labeled data. Since general-purpose datasets often lack specific Lithuanian nuances (like 'šakotis' vs generic spit cakes), we are building a ground-up dataset using:
- **Wikimedia Commons**: For high-quality, license-safe imagery.
- **Gemini 2.5 Flash**: To generate complex, expert-level Lithuanian QA pairs.
- **Human-in-the-loop**: A manual review stage to ensure 100% accuracy before training.

## 1. Setup

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!pip -q install requests pillow imagehash tqdm google-generativeai ipywidgets pandas

In [ ]:
import os, json, time, hashlib, io, re
from pathlib import Path
from typing import Any
from google.colab import userdata
import google.generativeai as genai
import requests
from PIL import Image
import google.generativeai as genai
import imagehash
from tqdm.auto import tqdm
import pandas as pd
import random

# Project paths on Drive (everything persists across Colab disconnects)
PROJECT_DIR = Path("/content/drive/MyDrive/VU_DL_task2")
IMAGES_DIR = PROJECT_DIR / "images"
META_DIR = PROJECT_DIR / "meta"
QA_DIR = PROJECT_DIR / "qa"
DATASET_DIR = PROJECT_DIR / "dataset"
for d in [IMAGES_DIR, META_DIR, QA_DIR, DATASET_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_DIR)

In [ ]:
# Load Gemini API key from Colab Secrets (sidebar key icon -> add GEMINI_API_KEY)

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)
print("Gemini configured.")

## 2. Categories

10 Lithuanian cultural categories. Each maps to a Wikimedia Commons category name and a target image count.
Feel free to edit the list before running the scraper.

In [ ]:
CATEGORIES = [
    # (label, [commons_category aliases tried in order], target_count, description_lt)
    (
        "cepelinai",
        ["Cepelinai"],
        25,
        "tradicinis lietuviškas bulvinių kukulių patiekalas su mėsos įdaru",
    ),
    (
        "sakotis",
        ["Baumkuchen", "Spit cakes", "Šakotis"],
        25,
        "tradicinis lietuviškas šakotas tortas, kepamas ant besisukančio iešmo",
    ),
    (
        "kibinai",
        ["Kibinai", "Kybyn", "Karaite cuisine"],
        25,
        "karaimų virtuvės pyragėliai su mėsos įdaru",
    ),
    ("gediminas_tower", ["Gediminas Tower"], 25, "Gedimino pilies bokštas Vilniuje"),
    (
        "trakai_castle",
        ["Trakai Island Castle"],
        25,
        "Trakų salos pilis ant Galvės ežero",
    ),
    ("hill_of_crosses", ["Hill of Crosses"], 25, "Kryžių kalnas netoli Šiaulių"),
    (
        "vytis",
        ["Vytis", "Coats of arms of Lithuania"],
        20,
        "Vytis - Lietuvos valstybės herbas, raitelis su kalaviju ir skydu",
    ),
    (
        "gates_of_dawn",
        ["Gate of Dawn", "Aušros Vartai"],
        25,
        "Aušros Vartai - garsus Vilniaus senamiestio vartai su stebuklinguoju Dievo Motinos paveikslu",
    ),
    (
        "kaziuko_muge",
        ["Kaziuko mugė", "Saint Casimir's Fair", "Kaziukas Fair"],
        25,
        "Kaziuko mugė - tradicinė pavasario mugė Vilniuje, šv. Kazimiero garbei",
    ),
    (
        "curonian_spit",
        ["Curonian Spit", "Nida", "Kuršių nerija"],
        25,
        "Kuršių nerija - UNESCO sąraše esanti smiltynė, žinoma dėl kopų ir žvejų kaimų",
    ),
]
TOTAL_TARGET = sum(c[2] for c in CATEGORIES)
print(f"Total target images: {TOTAL_TARGET}")

## 3. Wikimedia Commons scraper

### Methodology:
1. **License Filtering**: We only accept CC0, CC-BY, and Public Domain images to ensure the dataset is legally redistributable.
2. **Perceptual Hashing (pHash)**: Unlike MD5, which changes with a single pixel shift, pHash identifies 'visually similar' images. This prevents near-duplicate images from inflating our metrics or causing overfitting.
3. **Thumbnail Optimization**: Images are resized to 1024px to balance visual detail with the input constraints of the Qwen2-VL model.

In [ ]:
COMMONS_API = "https://commons.wikimedia.org/w/api.php"
USER_AGENT = "VU-DL-Task2-Dataset/1.0 (educational; contact: student@vu.lt)"
ALLOWED_LICENSES_KEYWORDS = ["cc0", "cc by", "cc-by", "public domain", "pd-"]
BLOCKED_MIMES = {"image/svg+xml", "image/svg", "image/x-icon"}


def list_category_files(category: str, limit: int = 80) -> list[str]:
    """Return up to ``limit`` File: page titles inside a Commons category.

    Args:
        category: Wikimedia Commons category name.
        limit: Maximum number of file titles to retrieve.

    Returns:
        A list of ``File:`` page titles found in the category.
    """
    titles, cont = [], None
    while len(titles) < limit:
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": f"Category:{category}",
            "cmtype": "file",
            "cmlimit": min(50, limit - len(titles)),
            "format": "json",
        }
        if cont:
            params["cmcontinue"] = cont
        r = requests.get(
            COMMONS_API, params=params, headers={"User-Agent": USER_AGENT}, timeout=30
        )
        r.raise_for_status()
        data = r.json()
        titles.extend(
            [m["title"] for m in data.get("query", {}).get("categorymembers", [])]
        )
        cont = data.get("continue", {}).get("cmcontinue")
        if not cont:
            break
    return titles


def get_file_info(title: str) -> dict[str, Any] | None:
    """Return metadata dict for a single Commons file page.

    Args:
        title: The ``File:`` page title on Wikimedia Commons.

    Returns:
        A dict with keys ``url``, ``mime``, ``license_short``, ``author``,
        etc., or ``None`` if the page cannot be resolved.
    """
    params = {
        "action": "query",
        "titles": title,
        "prop": "imageinfo",
        "iiprop": "url|extmetadata|mime|size",
        "iiurlwidth": 1024,
        "format": "json",
    }
    r = requests.get(
        COMMONS_API, params=params, headers={"User-Agent": USER_AGENT}, timeout=30
    )
    r.raise_for_status()
    pages = r.json().get("query", {}).get("pages", {})
    for _, page in pages.items():
        ii = page.get("imageinfo", [{}])[0]
        meta = ii.get("extmetadata", {})
        thumb = ii.get("thumburl") or ii.get("url")
        return {
            "title": title,
            "url": thumb,
            "mime": ii.get("thumbmime", ii.get("mime", "")),
            "width": ii.get("thumbwidth", ii.get("width", 0)),
            "height": ii.get("thumbheight", ii.get("height", 0)),
            "license_short": meta.get("LicenseShortName", {}).get("value", "").lower(),
            "license_url": meta.get("LicenseUrl", {}).get("value", ""),
            "author": re.sub(
                "<[^>]+>", "", meta.get("Artist", {}).get("value", "")
            ).strip(),
            "description_en": re.sub(
                "<[^>]+>", "", meta.get("ImageDescription", {}).get("value", "")
            ).strip()[:500],
        }
    return None


def license_ok(license_short: str) -> bool:
    """Check whether a license string matches any of the allowed keywords.

    Args:
        license_short: Lowercased short license name from Wikimedia metadata.

    Returns:
        ``True`` if the license is permissive enough for redistribution.
    """
    return any(k in license_short for k in ALLOWED_LICENSES_KEYWORDS)


def list_files_with_aliases(
    aliases: list[str], limit: int = 80
) -> tuple[list[str], str]:
    """Try each alias; return titles from the first one that yields results.

    Args:
        aliases: Ordered list of Commons category names to try.
        limit: Maximum number of file titles to retrieve.

    Returns:
        A tuple of ``(file_titles, used_alias)``.
    """
    for alias in aliases:
        titles = list_category_files(alias, limit=limit)
        if titles:
            print(f'  using Commons category: "{alias}" ({len(titles)} files)')
            return titles, alias
        print(f'  alias "{alias}" -> 0 files')
    return [], aliases[0]


def download_image_bytes(url: str, max_retries: int = 3) -> bytes:
    """Download raw image bytes with retry and exponential backoff.

    Args:
        url: Direct URL to the image file.
        max_retries: Number of retry attempts on transient errors.

    Returns:
        Raw image file bytes.

    Raises:
        ValueError: If the response content-type is not an image or is blocked.
        RuntimeError: If all retry attempts are exhausted.
    """
    for attempt in range(max_retries + 1):
        try:
            resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30)
            if resp.status_code in (429, 503, 500):
                time.sleep(2**attempt)
                continue
            resp.raise_for_status()
            ctype = resp.headers.get("Content-Type", "").split(";")[0].strip()
            if not ctype.startswith("image/"):
                raise ValueError(f"non-image content-type: {ctype}")
            if ctype in BLOCKED_MIMES:
                raise ValueError(f"blocked mime: {ctype}")
            return resp.content
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            if attempt < max_retries:
                time.sleep(2**attempt)
                continue
            raise
    raise RuntimeError(f"Failed after {max_retries} retries: {url}")

In [ ]:
def download_for_category(
    label: str,
    commons_aliases: list[str],
    target_count: int,
    description_lt: str,
) -> None:
    """Download up to ``target_count`` license-safe, deduplicated images for one category.

    Images are saved as JPEG thumbnails (max 1024 px) and metadata is appended
    to a per-category JSONL file.  Perceptual hashing is used to skip
    near-duplicate images.

    Args:
        label: Short category label used for directory and file naming.
        commons_aliases: Ordered Wikimedia Commons category aliases to try.
        target_count: Desired number of images to collect.
        description_lt: Lithuanian description of the category.
    """
    cat_dir = IMAGES_DIR / label
    cat_dir.mkdir(parents=True, exist_ok=True)
    meta_path = META_DIR / f"{label}.jsonl"
    seen_hashes: set[str] = set()
    rows: list[dict[str, Any]] = []

    existing = list(cat_dir.glob("*.jpg"))
    if len(existing) >= target_count:
        print(f"[{label}] already has {len(existing)} images, skipping.")
        return

    print(f"[{label}] resolving Commons category...")
    titles, used_alias = list_files_with_aliases(
        commons_aliases, limit=target_count * 6
    )
    if not titles:
        print(
            f"[{label}] NO files found in any alias - SKIPPING. Edit CATEGORIES to add more aliases."
        )
        return

    skip_counts: dict[str, int] = {
        "no_info": 0,
        "bad_mime": 0,
        "too_small": 0,
        "bad_license": 0,
        "dup": 0,
        "download_err": 0,
    }
    saved = len(existing)
    for title in tqdm(titles, desc=label):
        if saved >= target_count:
            break
        time.sleep(0.4)  # ALWAYS pace, including on failures, to avoid rate limiting
        try:
            info = get_file_info(title)
            if not info or not info["url"]:
                skip_counts["no_info"] += 1
                continue
            if not info["mime"].startswith("image/") or info["mime"] in BLOCKED_MIMES:
                skip_counts["bad_mime"] += 1
                continue
            if info["width"] < 400 or info["height"] < 400:
                skip_counts["too_small"] += 1
                continue
            if not license_ok(info["license_short"]):
                skip_counts["bad_license"] += 1
                continue
            img_bytes = download_image_bytes(info["url"])
            img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            phash = str(imagehash.phash(img))
            if phash in seen_hashes:
                skip_counts["dup"] += 1
                continue
            seen_hashes.add(phash)
            img.thumbnail((1024, 1024))
            fname = hashlib.md5(title.encode()).hexdigest()[:12] + ".jpg"
            img.save(cat_dir / fname, "JPEG", quality=88)
            rows.append(
                {
                    "image_path": str(cat_dir / fname),
                    "category": label,
                    "category_description_lt": description_lt,
                    "commons_title": title,
                    "commons_category_used": used_alias,
                    "source_url": info["url"],
                    "license": info["license_short"],
                    "license_url": info["license_url"],
                    "author": info["author"][:200],
                    "phash": phash,
                }
            )
            saved += 1
        except Exception:
            skip_counts["download_err"] += 1
            continue

    with open(meta_path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"[{label}] saved {saved}/{target_count}  skipped: {skip_counts}")


for label, aliases, target, desc in CATEGORIES:
    download_for_category(label, aliases, target, desc)

### Manually upload šakotis images

In [ ]:
# --- Register manually-uploaded images for specified categories ---
# Edit this list: add any category where you replaced/added images by hand.
MANUAL_CATEGORIES = ["sakotis"]

# Build a lookup: label -> description_lt from the CATEGORIES list
cat_desc = {label: desc for label, _, _, desc in CATEGORIES}

for label in MANUAL_CATEGORIES:
    cat_dir = IMAGES_DIR / label
    meta_path = META_DIR / f"{label}.jsonl"

    # Load existing metadata paths for this category
    existing_paths = set()
    if meta_path.exists():
        with open(meta_path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    existing_paths.add(json.loads(line)["image_path"])

    # Find image files in the folder that are NOT yet in metadata
    new_images = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        for img_file in sorted(cat_dir.glob(ext)):
            img_path_str = str(img_file)
            if img_path_str not in existing_paths:
                new_images.append(img_file)

    if not new_images:
        print(f"[{label}] no new manually-uploaded images found.")
        continue

    # Resize to max 1024px and re-save as JPG (consistent with scraped images)
    rows = []
    for img_file in tqdm(new_images, desc=f"{label} (manual)"):
        img = Image.open(img_file).convert("RGB")
        phash = str(imagehash.phash(img))
        img.thumbnail((1024, 1024))

        # Rename to .jpg if needed, using content hash for uniqueness
        fname = hashlib.md5(img_file.name.encode()).hexdigest()[:12] + ".jpg"
        out_path = cat_dir / fname
        if out_path != img_file:
            img.save(out_path, "JPEG", quality=88)
            # Remove original if it was a different file
            if img_file.exists() and img_file != out_path:
                img_file.unlink()
        else:
            img.save(out_path, "JPEG", quality=88)

        rows.append(
            {
                "image_path": str(out_path),
                "category": label,
                "category_description_lt": cat_desc.get(label, ""),
                "commons_title": f"Manual upload: {img_file.name}",
                "commons_category_used": "manual_upload",
                "source_url": "",
                "license": "own work / manual upload",
                "license_url": "",
                "author": "student",
                "phash": phash,
            }
        )

    with open(meta_path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"[{label}] registered {len(rows)} manually-uploaded images.")

# Also clean up: remove metadata entries whose image files no longer exist
for label in MANUAL_CATEGORIES:
    meta_path = META_DIR / f"{label}.jsonl"
    if not meta_path.exists():
        continue
    kept = []
    removed = 0
    with open(meta_path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            entry = json.loads(line)
            if Path(entry["image_path"]).exists():
                kept.append(line.strip())
            else:
                removed += 1
    if removed:
        with open(meta_path, "w", encoding="utf-8") as f:
            f.write("\n".join(kept) + "\n")
        print(f"[{label}] removed {removed} stale metadata entries (files deleted).")

In [ ]:
# Consolidate metadata
all_meta = []
for f in META_DIR.glob("*.jsonl"):
    with open(f, encoding="utf-8") as fh:
        all_meta.extend(json.loads(line) for line in fh if line.strip())
meta_df = pd.DataFrame(all_meta).drop_duplicates(subset="image_path")
meta_df.to_csv(META_DIR / "all_images.csv", index=False)
print(meta_df.groupby("category").size())
print("Total images:", len(meta_df))

## 4. Generate Lithuanian QA pairs with Gemini

### Prompt Design Strategy:
We use a 4-tier QA structure to teach the model different aspects of vision and culture:
1. **Identification**: Basic classification (What is this?).
2. **Visual detail**: Forces the model to attend to specific pixels (e.g., the spikes on a šakotis).
3. **Cultural context**: Adds depth, teaching the model 'why' an object is significant in Lithuania.
4. **Adversarial**: This is the most critical. By explicitly asking questions that usually confuse VLMs (e.g., distinguishing 'Kibinai' from 'Empanadas'), we train the model to be more discriminative.

In [ ]:
GEMINI_MODEL = "gemini-2.5-flash"
model = genai.GenerativeModel(GEMINI_MODEL)

QA_PROMPT_TEMPLATE = """Tu esi lietuvių kultūros ir kalbos ekspertas. Pateiktas vaizdas iš kategorijos: "{category_desc}".

Sugeneruok TIKSLIAI 4 klausimus ir atsakymus LIETUVIŲ kalba apie šį konkretų vaizdą:
1. identification: Kas pavaizduota? (trumpas faktinis atsakymas)
2. visual: Kokia konkreti vizuali detalė matoma šiame vaizde? (turi remtis tuo, ką realiai matai)
3. cultural: Koks kultūrinis arba istorinis kontekstas? (1-2 sakiniai)
4. adversarial: Klausimas, į kurį bendrinis modelis greičiausiai atsakytų neteisingai (pvz. supainiotų su panašiu užsienio reiškiniu) - parašyk teisingą atsakymą.

Grąžink validų JSON tokios formos:
{{"qa": [
  {{"type": "identification", "q": "...", "a": "..."}},
  {{"type": "visual",         "q": "...", "a": "..."}},
  {{"type": "cultural",       "q": "...", "a": "..."}},
  {{"type": "adversarial",    "q": "...", "a": "..."}}
]}}"""


def generate_qa_for_image(
    image_path: str | Path,
    category_desc: str,
    max_retries: int = 5,
) -> list[dict[str, str]] | None:
    """Generate four Lithuanian QA pairs for a single image using Gemini.

    The prompt requests identification, visual detail, cultural context,
    and adversarial question types.

    Args:
        image_path: Filesystem path to the source image.
        category_desc: Lithuanian description of the image category.
        max_retries: Maximum number of API call attempts with exponential backoff.

    Returns:
        A list of four QA dicts (keys: ``type``, ``q``, ``a``), or ``None``
        if generation fails after all retries.
    """
    img = Image.open(image_path).convert("RGB")
    prompt = QA_PROMPT_TEMPLATE.format(category_desc=category_desc)

    for attempt in range(max_retries):
        try:
            # 2. Use Native JSON mode
            resp = model.generate_content(
                [prompt, img],
                generation_config={
                    "temperature": 0.4,
                    "response_mime_type": "application/json",  # Forces clean JSON output
                },
            )
            data = json.loads(resp.text)

            if data and isinstance(data.get("qa"), list) and len(data["qa"]) == 4:
                return data["qa"]

        except Exception as e:
            wait_time = 2**attempt
            print(
                f"  [API Error] Retry {attempt + 1}/{max_retries} in {wait_time}s: {e}"
            )
            time.sleep(wait_time)

    return None


QA_RAW_PATH = QA_DIR / "qa_raw.jsonl"

# Resume support: skip images we already processed
done_paths: set[str] = set()
if QA_RAW_PATH.exists():
    with open(QA_RAW_PATH, encoding="utf-8") as f:
        for line in f:
            try:
                done_paths.add(json.loads(line)["image_path"])
            except Exception:
                pass
print(f"Already generated for {len(done_paths)} images.")

with open(QA_RAW_PATH, "a", encoding="utf-8") as out:
    for _, row in tqdm(meta_df.iterrows(), total=len(meta_df)):
        if row["image_path"] in done_paths:
            continue

        qa = generate_qa_for_image(row["image_path"], row["category_description_lt"])

        if not qa:
            print(f'\n  [FATAL] Failed after 5 retries: {row["image_path"]}')
            continue

        out.write(
            json.dumps(
                {
                    "image_path": row["image_path"],
                    "category": row["category"],
                    "qa": qa,
                },
                ensure_ascii=False,
            )
            + "\n"
        )
        out.flush()

## 5. Manual review UI

Inline ipywidgets reviewer. For each image: see the picture, edit any QA pair, mark each as **accept / edit / reject**. Decisions are saved continuously to `qa_reviewed.jsonl` so we can stop and resume.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import traceback

QA_REVIEWED_PATH = QA_DIR / "qa_reviewed.jsonl"

# Load raw + previously reviewed
raw_items = [json.loads(l) for l in open(QA_RAW_PATH, encoding="utf-8") if l.strip()]
reviewed_paths: set[str] = set()
if QA_REVIEWED_PATH.exists():
    reviewed_paths = {
        json.loads(l)["image_path"]
        for l in open(QA_REVIEWED_PATH, encoding="utf-8")
        if l.strip()
    }
pending = [r for r in raw_items if r["image_path"] not in reviewed_paths]

state: dict[str, int] = {"idx": 0}


def render() -> None:
    """Render the interactive QA review widget for the current pending image.

    Displays the image, editable QA text areas, and accept/skip/reject
    buttons.  State is persisted to ``qa_reviewed.jsonl`` on each action.
    """
    # 1. Clear the entire Colab cell directly (No Output wrapper needed)
    clear_output(wait=True)

    # Show remaining progress at the top of the cell
    print(f"Pending review: {len(pending) - state['idx']} / {len(raw_items)}")

    try:
        if state["idx"] >= len(pending):
            print("All done! Dataset is ready.")
            return

        item = pending[state["idx"]]
        print(
            f"[{state['idx']+1}/{len(pending)}]  {item.get('category', 'unknown')}  {Path(item.get('image_path', '')).name}"
        )

        # 2. Display image directly to the cell
        display(Image.open(item["image_path"]).copy().resize((384, 384)))

        text_areas: list[
            tuple[str, widgets.Textarea, widgets.Textarea, widgets.Checkbox]
        ] = []
        qa_list = item.get("qa", [])
        if isinstance(qa_list, dict):
            qa_list = [qa_list]

        for i, qa in enumerate(qa_list):
            q_type = qa.get("type", f"unknown_{i}")
            q_val = qa.get("q", "ERROR: Missing Question")
            a_val = qa.get("a", "ERROR: Missing Answer")

            ta_q = widgets.Textarea(
                value=q_val,
                description=f"Q ({q_type})",
                layout=widgets.Layout(width="95%", height="50px"),
            )
            ta_a = widgets.Textarea(
                value=a_val,
                description="A",
                layout=widgets.Layout(width="95%", height="60px"),
            )
            keep = widgets.Checkbox(value=True, description="keep")
            text_areas.append((q_type, ta_q, ta_a, keep))

            # 3. Display inputs directly to the cell
            display(widgets.VBox([ta_q, ta_a, keep]))

        save_btn = widgets.Button(description="Save & Next", button_style="success")
        skip_btn = widgets.Button(description="Skip image", button_style="warning")
        reject_btn = widgets.Button(description="Reject all", button_style="danger")

        def on_save(_: Any) -> None:
            """Persist kept QA pairs and advance to the next image."""
            kept = [
                {"type": t, "q": q.value.strip(), "a": a.value.strip()}
                for (t, q, a, k) in text_areas
                if k.value and q.value.strip() and a.value.strip()
            ]
            with open(QA_REVIEWED_PATH, "a", encoding="utf-8") as f:
                f.write(
                    json.dumps(
                        {
                            "image_path": item["image_path"],
                            "category": item.get("category", "unknown"),
                            "qa": kept,
                        },
                        ensure_ascii=False,
                    )
                    + "\n"
                )
            state["idx"] += 1
            render()

        def on_skip(_: Any) -> None:
            """Skip the current image without saving."""
            state["idx"] += 1
            render()

        def on_reject(_: Any) -> None:
            """Reject all QA pairs for the current image and advance."""
            with open(QA_REVIEWED_PATH, "a", encoding="utf-8") as f:
                f.write(
                    json.dumps(
                        {
                            "image_path": item["image_path"],
                            "category": item.get("category", "unknown"),
                            "qa": [],
                        },
                        ensure_ascii=False,
                    )
                    + "\n"
                )
            state["idx"] += 1
            render()

        save_btn.on_click(on_save)
        skip_btn.on_click(on_skip)
        reject_btn.on_click(on_reject)

        # 4. Display buttons directly to the cell
        display(widgets.HBox([save_btn, skip_btn, reject_btn]))

    except Exception as e:
        # If it crashes, this will print the exact reason to our cell
        print(f"CRITICAL UI ERROR: {e}")
        print(traceback.format_exc())

        emerg_btn = widgets.Button(description="Emergency Skip", button_style="danger")

        def on_emerg(_: Any) -> None:
            """Emergency skip handler to recover from UI errors."""
            state["idx"] += 1
            render()

        emerg_btn.on_click(on_emerg)
        display(emerg_btn)


render()

## 6. Build final dataset (train / val / test)

### Splitting Strategy:
We utilize a **Stratified Split** based on categories.
- **Why?** If we did a random split, a rare category (like 'Vytis') might end up entirely in the training set, meaning we couldn't evaluate the model's performance on it.
- **Leakage Prevention**: We split by image path rather than QA pair. This ensures the model doesn't see a different question about the same image during testing.

In [ ]:
random.seed(42)

reviewed = [
    json.loads(l) for l in open(QA_REVIEWED_PATH, encoding="utf-8") if l.strip()
]
reviewed = [r for r in reviewed if r["qa"]]  # drop empty

# Drop entries whose image file no longer exists (e.g. old scraped sakotis images)
before = len(reviewed)
reviewed = [r for r in reviewed if Path(r["image_path"]).exists()]
if len(reviewed) < before:
    print(f"Dropped {before - len(reviewed)} entries with missing image files.")

# Group by category, then split images (not QA pairs) so no leakage
by_cat = {}
for r in reviewed:
    by_cat.setdefault(r["category"], []).append(r)

splits = {"train": [], "val": [], "test": []}
for cat, items in by_cat.items():
    random.shuffle(items)
    n = len(items)
    n_test = max(2, int(0.10 * n))
    n_val = max(2, int(0.10 * n))
    splits["test"].extend(items[:n_test])
    splits["val"].extend(items[n_test : n_test + n_val])
    splits["train"].extend(items[n_test + n_val :])

for k, v in splits.items():
    print(k, "images:", len(v), "qa pairs:", sum(len(x["qa"]) for x in v))

with open(DATASET_DIR / "dataset.jsonl", "w", encoding="utf-8") as f:
    for split_name, items in splits.items():
        for it in items:
            for qa in it["qa"]:
                f.write(
                    json.dumps(
                        {
                            "split": split_name,
                            "image_path": it["image_path"],
                            "category": it["category"],
                            "qa_type": qa["type"],
                            "question": qa["q"],
                            "answer": qa["a"],
                        },
                        ensure_ascii=False,
                    )
                    + "\n"
                )

with open(DATASET_DIR / "splits.json", "w", encoding="utf-8") as f:
    json.dump(
        {k: [it["image_path"] for it in v] for k, v in splits.items()},
        f,
        ensure_ascii=False,
        indent=2,
    )

print("\nDataset written to", DATASET_DIR / "dataset.jsonl")